# Parallelization + Reflection (Design Patterns)

**Week 2 — Agentic AI: Building Autonomous Intelligent Systems**

When an agent has several **independent** pieces of work to do, running them one after another wastes time — most of every LLM call is spent *waiting on the network*. The **Parallelization Design Pattern** runs those independent calls concurrently and merges the results. In this lab you pair it with the **Reflection Design Pattern**, where the agent critiques and improves its own output, to build an analysis pipeline that is both fast and self-correcting.

## Introduction

In this lesson we answer:

- What is the parallelization design pattern (fan-out / fan-in)?
- What use cases is it applied to?
- What building blocks are needed to implement it?
- How does `asyncio` actually make independent calls run concurrently?
- What considerations make a parallel, self-reflecting agent trustworthy?

## Learning Goals

After completing this lesson you will be able to:

- Define the parallelization pattern and explain **fan-out** and **fan-in**.
- Identify use cases where running LLM calls concurrently pays off.
- Use `asyncio` and `asyncio.gather()` to run independent agent calls at the same time.
- Add a **reflection loop** that scores and rewrites output against a quality threshold.
- Recognize the trust and safety considerations of concurrent, self-critiquing agents.

You will implement:

1. `call_expert(persona, topic)` — one async expert call
2. `fan_out(topic)` — run all experts concurrently
3. `fan_in(topic, outputs)` — synthesize the perspectives
4. `critique(document)` — score + weaknesses + improvements via **structured output**
5. `rewrite(document, feedback)` — apply the critique
6. `reflection_loop(...)` and `analyze_with_reflection(...)` — the full pipeline

This notebook runs in **Google Colab** on the **Gemini API**.

## What is the Parallelization Design Pattern?

Parallelization splits a task into **independent sub-tasks**, runs them **at the same time**, then **combines** the results. The two halves have names:

- **Fan-out** — dispatch N independent prompts concurrently (here: three expert personas analyze the same topic).
- **Fan-in** — aggregate the N results into one coherent output (a synthesis).

```
                          ┌──────────────┐
                     ┌──▶ │   Expert A   │──┐
                     │    └──────────────┘  │
   topic ─fan-out────┼──▶ ┌──────────────┐  ├─fan-in─▶ synthesis
   (one input)       │    │   Expert B   │  │         (one document
                     │    └──────────────┘  │          combining all views)
                     └──▶ ┌──────────────┐  │
                          │   Expert C   │──┘
                          └──────────────┘
        all three run concurrently
        (~time of the slowest call, not the sum of all)
```

The speedup is real because each LLM call is mostly *waiting on the network* — concurrency lets those waits overlap.

## Use cases

The pattern fits any time you have independent work that does not depend on each other's results:

- **Multi-perspective analysis** — several expert "lenses" on the same input (this lab).
- **Map over a list** — summarize / classify / translate many documents at once.
- **Ensembling** — ask the same question several ways, then vote or merge.
- **Independent sub-questions** — research several facets of a topic in parallel, then synthesize.
- **Concurrent guardrails** — run safety, policy, and quality checks alongside the main answer.

If a sub-task depends on another's output (step 2 needs step 1's result), that is **chaining**, not parallelization — run those sequentially.

## Building blocks

To implement the pattern you need:

- **Independent units of work** — each sub-task must not depend on the others' outputs.
- **An async call function** — a coroutine that performs one unit (here, one expert call) and can be `await`-ed.
- **A concurrency primitive** — `asyncio.gather()` to launch many coroutines and await them together (fan-out).
- **An aggregator** — logic that merges the parallel results into one output (fan-in); often itself an LLM call.
- **(For reflection)** a structured **critic**, a **rewrite** step, and a **stopping rule** (quality threshold or max iterations).
- **Rate-limit handling** — a retry/backoff so a burst of concurrent calls degrades gracefully instead of crashing.

## Considerations for trustworthy agents

- **Rate limits and cost** — fanning out multiplies requests-per-second and total tokens. Bound concurrency and retry on `429` (this lab does both).
- **Failure isolation** — one sub-task failing should not sink the batch. Capture errors per task rather than letting one exception abort everything.
- **Ordering is nondeterministic** — `gather` preserves *input* order, but results finish at unpredictable times; never rely on which finishes first.
- **Reflection honesty** — a model grading its own work tends to be over-generous. Use a concrete rubric and a hard iteration cap so the loop always terminates.
- **Synthesis faithfulness** — fan-in should represent the sources, not invent consensus. Keep genuine disagreement visible.

## The Reflection Design Pattern

Once fan-in produces a draft, the agent improves it by **critiquing its own output and rewriting** — a loop that continues until the work clears a quality bar or hits an iteration limit.

```
   draft
     │
     ▼
   ┌──────────────┐
   │   critique   │  ── returns score + weaknesses + improvements (structured data)
   └──────────────┘
     │
     ▼
   score >= threshold ?  ──no──▶  rewrite ──┐
     │ yes                                   │  loop, up to max_iterations
     ▼                                       │
   final document  ◀───────────────────────-┘
```

Because the critique returns **structured** data (a numeric score and lists), the loop makes a real decision instead of parsing prose.

## A primer on asyncio (read this before the code)

The entire speedup in this lab comes from `asyncio`. It is worth understanding *why* it works.

### Concurrency is not parallelism
`asyncio` runs on a **single thread**. It does **not** use multiple CPU cores. What it does is **interleave** tasks while they wait on I/O. An LLM call spends almost all its time waiting for the network to reply — during that wait, asyncio lets other calls start and wait too. The waits overlap, so wall-clock time collapses. (For CPU-bound work — heavy math — asyncio does nothing useful; you would reach for multiprocessing.)

### Coroutines, await, and the event loop
- `async def` defines a **coroutine** — a function you can pause. Calling it returns a coroutine object; it does **not** run until awaited.
- `await` runs a coroutine and suspends the caller until it finishes, **yielding control to the event loop** so other coroutines can make progress.
- The **event loop** is the scheduler that juggles all the suspended coroutines, resuming each when its I/O is ready.

### Sequential vs concurrent
```
SEQUENTIAL  (await one call, then the next -- the waits stack up)
  A  |==== waiting on network ====| done
  B                                |==== waiting ====| done
  C                                                   |==== waiting ====| done
     +--------------+--------------+--------------+    total ~ 3x one call

CONCURRENT  (asyncio.gather -- all launched together, waits overlap)
  A  |==== waiting on network ====| done
  B  |==== waiting on network ====| done
  C  |==== waiting on network ====| done
     +--------------+    total ~ 1x one call
```

`asyncio.gather(*coros)` schedules every coroutine on the loop at once and returns their results **in input order** once all finish. That is exactly fan-out.

### Top-level await in Colab
Colab and Jupyter already run an event loop, so you can write `await some_coroutine()` directly in a cell — and this lab does. In a plain `.py` script there is no running loop, so you would wrap the entry point in `asyncio.run(main())`. Calling `asyncio.run(...)` inside Colab raises *"event loop is already running"*, so we never do.

## Setup: add your Gemini API key as a Colab secret

1. Get a key from [Google AI Studio](https://aistudio.google.com/app/apikey).
2. In Colab, click the **🔑 key icon** in the left sidebar ("Secrets").
3. Add a new secret named **`GEMINI_API_KEY`** and paste your key as the value.
4. Toggle **"Notebook access"** on for that secret.

> **Rate-limit note:** this lab fans out several calls at once and loops over reflections, so the helpers below automatically wait and retry if you hit a rate limit (HTTP 429). Runs will simply pause briefly rather than crash.

The next cell installs the SDK and reads the key from your Colab secrets.

In [161]:
!pip install -q google-genai python-dotenv

In [162]:
import os
import time
import asyncio

from google.colab import userdata
from dotenv import load_dotenv
from google import genai
from google.genai import errors, types
from pydantic import BaseModel

load_dotenv()

api_key = userdata.get('GEMINI_API_KEY')
if not api_key:
    raise RuntimeError("Missing GEMINI_API_KEY. Add it in Colab Secrets (🔑 icon) and enable notebook access.")

client = genai.Client(api_key=api_key)
MODEL = "gemini-3.1-flash-lite"

# Quick connectivity check
response = client.models.generate_content(
    model=MODEL,
    contents="Say 'Setup complete!' and nothing else.",
)
print(response.text)

Setup complete!


## The async helpers

The synchronous `client.models.generate_content` has an async twin: `client.aio.models.generate_content`. We `await` it. Because every expert call is independent, awaiting them inside `asyncio.gather()` lets them run **concurrently** instead of blocking one at a time.

- `acall(prompt, system=None)` returns prose (an analysis, a synthesis). The persona instructions go in `config=types.GenerateContentConfig(system_instruction=...)` — Gemini's equivalent of Anthropic's separate `system` argument.
- `acall_json(prompt, schema, system=None)` returns **structured data**. We set `response_mime_type="application/json"` and a `response_schema`, and read `r.parsed` — typed Python objects, guaranteed schema-valid. No `json.loads`, no regex, no "the model wrapped it in ```json fences" bugs.

Both route through `_acall()`, which retries once on a rate limit (HTTP 429).

In [163]:
# Three small helpers wrap the Gemini SDK so the rest of the lab stays readable.

# _acall is the low-level call the two public helpers below share. It sends one
# request to Gemini and, if we hit a rate limit (HTTP 429), waits 60 seconds and
# tries once more instead of crashing the whole run.
async def _acall(contents, config):
    """Make one async Gemini call; retry once after 60s on a 429 rate limit."""
    for attempt in range(2):
        try:
            return await client.aio.models.generate_content(
                model=MODEL, contents=contents, config=config
            )
        except errors.ClientError as e:
            # 429 = too many requests. Back off once, then re-raise if it happens again.
            if e.code == 429 and attempt == 0:
                print("  Rate limited. Waiting 60s, then retrying...")
                await asyncio.sleep(60)
            else:
                raise


# acall: use this when you want plain TEXT (prose) back -- an analysis, a synthesis.
# `system` holds the persona/instructions and is passed to Gemini as a system_instruction.
async def acall(prompt: str, system: str | None = None) -> str:
    """Send one prompt to Gemini (async) and return the response text."""
    config = types.GenerateContentConfig(system_instruction=system) if system else None
    r = await _acall(prompt, config)
    return (r.text or "").strip()


# acall_json: use this when you want STRUCTURED data back instead of prose.
# We hand Gemini a `schema` (a pydantic model) and switch on JSON mode, so the reply
# is validated into a typed Python object for us -- no manual parsing, no broken JSON.
async def acall_json(prompt: str, schema, system: str | None = None):
    """Generate schema-validated JSON (async); returns a typed object matching `schema`."""
    config = types.GenerateContentConfig(
        system_instruction=system,
        response_mime_type="application/json",
        response_schema=schema,
    )
    r = await _acall(prompt, config)
    return r.parsed

## The three experts

Each expert is its **own function** — `technical_expert`, `business_strategist`, `ethics_researcher` and `ux_researcher` — so you can read each one top to bottom with no shared list or loop to untangle. They all follow the *same* task (`analysis_prompt`) and differ only in their **persona** (the senior-expert `# Role` in the system text). That diversity of perspectives is what makes fanning out worthwhile — a single prompt would blend them into mush.

```
   topic
     │
     ├──▶ technical_expert(topic)       → software-engineering lens
     │
     ├──▶ business_strategist(topic)    → commercial / ROI lens
     │
     └──▶ ethics_researcher(topic)      → fairness / safety lens
     │
     └──▶ ux_researcher(topic)          → product userfriendlyness lens

   fan_out runs all four concurrently with asyncio.gather(...),
   then fan_in merges the three results into one synthesis.
```

In [164]:
# analysis_prompt builds the task text every expert shares.
# Only the persona (the system text in each function below) changes.
def analysis_prompt(topic: str) -> str:
    """Return the shared analysis instructions, with `topic` filled in."""
    return f"""# Context
You are one of several expert reviewers analyzing the same topic. Your analysis is later synthesized with the others, so your distinct, in-depth perspective is what matters most.

# Task
Analyze the topic in the Input section from your area of expertise.

# Constraints
- 150-200 words.
- Be specific and concrete: name real techniques, mechanisms, trade-offs, or examples.
- Stay in your lane -- argue from your perspective, not a balanced overview.

# Format
One focused paragraph. No headings, no bullet lists.

# Input (topic)
{topic}"""


# Expert 1 -- the technical-implementation perspective.
async def technical_expert(topic: str) -> str:
    """One expert call: analyze the topic as a senior software engineer."""
    system = """# Role
You are a senior staff software engineer with deep, hands-on expertise in building and shipping AI systems.

# Perspective
Analyze through a technical-implementation lens: architectures, techniques, data and infra needs, failure modes, and the hard engineering challenges."""
    return await acall(analysis_prompt(topic), system=system)


# Expert 2 -- the commercial perspective.
async def business_strategist(topic: str) -> str:
    """One expert call: analyze the topic as a senior business strategist."""
    system = """# Role
You are a senior business strategist and former operator who evaluates technology commercially.

# Perspective
Analyze in terms of market opportunity, ROI, adoption risk, costs, and competitive dynamics. Be concrete about the business implications."""
    return await acall(analysis_prompt(topic), system=system)


# Expert 3 -- the societal-impact perspective.
async def ethics_researcher(topic: str) -> str:
    """One expert call: analyze the topic as a senior AI ethics researcher."""
    system = """# Role
You are a senior AI ethics researcher focused on real-world societal impact.

# Perspective
Analyze through fairness, safety, accountability, transparency, and long-term consequences. Name concrete risks and who bears them."""
    return await acall(analysis_prompt(topic), system=system)

# Expert 4 -- the product userfriendlyness perspective.
async def ux_researcher(topic: str) -> str:
    """One expert call: analyze the topic as a senior UX researcher."""
    system = """# Role
You are a senior UX researcher focused on how is the product user friendly and does it satisfy all the needs of user.

# Perspective
Analyze through usability, product design , gathering data, user expectation, any bugs."""
    return await acall(analysis_prompt(topic), system=system)

## Step 1 — Fan-out: run the experts concurrently

`fan_out` calls the three expert functions and hands them to `asyncio.gather(...)`, which runs them **concurrently** and returns their results in order. We pass the three calls in **explicitly** — no list, no loop — so it is obvious exactly what is running in parallel. If each call takes ~3s, running them one at a time would take ~9s; fanned out, they finish in about ~3s.

We never call `asyncio.run` — these are `async def` functions, so we drive them later with top-level `await`.

In [165]:
# fan_out runs the three expert calls at the SAME time with asyncio.gather,
# then returns their three analyses (in the order they were passed in).
async def fan_out(topic: str):
    """Run the four experts concurrently; return (technical, business, ethics, ux)."""
    print("Fanning out to 4 experts in parallel...")
    technical, business, ethics, ux = await asyncio.gather(
        technical_expert(topic),
        business_strategist(topic),
        ethics_researcher(topic),
        ux_researcher(topic),
    )
    print("  All 4 expert responses received.")
    return technical, business, ethics, ux

## Step 2.1 — Plain Fan-in: synthesize the perspectives

Fan-out gives us four separate analyses; fan-in collapses them into one document. We pass the three analyses in by name, format them into the prompt, and ask the model to integrate them, surface agreement and tension, and end with a key insight. This is plain prose, so we use `acall(...)`.

In [166]:
# Strategy1 : Plain fan_in takes the four analyses by name and asks the model to merge them into one briefing.
async def plain_fan_in(topic: str, technical: str, business: str, ethics: str, ux: str) -> str:
    """Merge the four expert analyses into one synthesized briefing string."""
    perspectives = (
        f"### Technical Expert\n{technical}\n\n"
        f"### Business Strategist\n{business}\n\n"
        f"### Ethics Researcher\n{ethics}\n\n"
        f"### UX Researcher\n{ux}"
    )
    synthesis = await acall(
        f"""# Context
You are given four independent expert analyses of the same topic, each written from a different perspective.

# Task
Synthesize them into one coherent briefing on the topic.

# Constraints
- 300-400 words.
- Integrate all three perspectives; do not merely concatenate them.
- Make areas of agreement and tension explicit; do not manufacture false consensus.
- End the final section with a single, sharp key insight.

# Format
Flowing prose, ending with a one-sentence key insight. No bullet lists.

# Topic
{topic}

# Expert analyses
{perspectives}""",
        system="# Role\nYou are a senior editorial synthesist who turns multiple expert views into one sharp, faithful briefing.",
    )
    word_count = len(synthesis.split())
    print(f"  Fan-in complete. Synthesis: {word_count} words.")
    return synthesis

## Step 2.2 — Fan-in: Structured Synthesis

In [167]:
# Strategy 2: Structured Merge (Isolated Domain Sections)
async def structured_synthesis(topic: str, technical: str, business: str, ethics: str, ux: str) -> str:
    """Combines perspectives by dedicating a clear, isolated flowing section to each domain."""
    perspectives = (
        f"### Technical Expert\n{technical}\n\n"
        f"### Business Strategist\n{business}\n\n"
        f"### Ethics Researcher\n{ethics}\n\n"
        f"### UX Researcher\n{ux}"
    )

    synthesis = await acall(f"""

    #Context
    You are given four independent expert analyses of the same topic, each written from a different perspective.

    # Task
    Synthesize the four expert reports into a clean, structured briefing.
    You must present this synthesis using flowing paragraphs under explicit bold headers for each domain:
    Structural Architecture, Business Perspective, Ethical View and User Experience.

    # Constraints
    - 350-450 words total.
    - No bullet points. Use flowing prose under each header.
    - End the final section with a single, sharp key insight.

    #Topic:
    {topic}

    # Expert analyses
    {perspectives}""",
          system="# Role\n You are a Senior Systems Architect who keeps the knowledge of all the positions and arranges the result in a very structural organised pattern",

    )
    print(f"  Structured Fan-in complete. Synthesis: {len(synthesis.split())} words.")
    return synthesis




## Step 2.3 — Fan-in: Consensus-vs-Divergence

In [168]:
# Strategy 3: Consensus-vs-Divergence Layout
async def consensus_divergence_synthesis(topic: str, technical: str, business: str, ethics: str, ux: str) -> str:
    """Synthesizes insights by highlighting where experts agree and where they clash."""
    perspectives = (
        f"### Technical Expert\n{technical}\n\n"
        f"### Business Strategist\n{business}\n\n"
        f"### Ethics Researcher\n{ethics}\n\n"
        f"### UX Researcher\n{ux}"
    )

    synthesis = await acall(f"""
    #Context
    You are given four independent expert analyses of the same topic, each written from a different perspective.

    # Task
    Review the four domain perspectives and synthesize them into a consensus-vs-divergence analysis.
    You must present this synthesis using flowing paragraphs under explicit bold headers:
    1. Consensus Area
    2. Divergence Area


    # Constraints
    - 350-450 words total.
    - Make areas of agreement and tension explicit; do not manufacture false consensus.
    - No bullet lists. Use clean paragraphs under the bold section titles.
    - End the final section with a single, sharp key insight.

    #Topic:
    {topic}

    # Expert analyses
    {perspectives}""",
        system="# Role\nYou are a Senior AI Systems Architect knowing the common and uncommon points over multiple domains",
    )
    print(f"  Consensus/Divergence Fan-in complete. Synthesis: {len(synthesis.split())} words.")
    return synthesis

## Step 3 — Reflection: structured critique

The reflection loop needs the critique as **data** — a numeric score to compare against a threshold, plus lists of weaknesses and improvements to feed into the rewrite. The original `.py` lab asked for JSON in free text and then scraped it with a regex and `json.loads`, with a `score: 5` fallback when parsing failed. That is fragile.

Instead we define a `Critique` pydantic model and let Gemini's structured-output mode guarantee a valid object. `acall_json(..., schema=Critique)` returns a typed `Critique`, so downstream code reads `feedback.score`, `feedback.weaknesses`, `feedback.improvements` by **attribute** — no parsing, no fallback needed.

Note the `REFLECTION_SYSTEM` prompt no longer dictates a JSON layout; the schema does that. It just describes the *editorial job*.

In [169]:
REFLECTION_SYSTEM = """# Role
You are a senior editor and critical reviewer of analytical writing. You are exacting: you surface the real weaknesses, not cosmetic ones, and you grade honestly rather than generously."""

REWRITE_SYSTEM = """# Role
You are a senior nonfiction editor who rewrites drafts to be tighter, more specific, and better supported, while preserving the author's substance and voice."""


# Critique is a pydantic model -- a typed schema for the LLM's output.
# We use pydantic so Gemini's JSON reply is validated into exactly this shape
# (an int `score` plus two string lists). If the model returns the wrong shape,
# pydantic raises here instead of letting malformed data flow into the loop below.
class Critique(BaseModel):
    """Structured reviewer feedback: a 1-10 score plus weaknesses and improvements."""
    score: int            # quality score, 1-10
    weaknesses: list[str]
    improvements: list[str]


# critique asks the model to grade the draft and returns a typed Critique
# (acall_json + the schema above) so the loop can compare score to a threshold.
async def critique(document: str) -> Critique:
    """Critique the document and return structured feedback (score, weaknesses, improvements)."""
    return await acall_json(
        f"""# Context
This document is a draft inside a reflection loop; your critique decides whether and how it gets rewritten, so be concrete and honest.

# Task
Critique the document in the Input section and score its quality.

# Constraints
- Score from 1 (poor) to 10 (excellent); reserve 9-10 for genuinely excellent work.
- Identify specific weaknesses: vague claims, missing evidence, logical gaps, repetition.
- Propose 2-3 concrete, actionable improvements.

# Format
Return structured data: an integer `score`, a list of `weaknesses`, and a list of `improvements`.

# Input (document)
{document}""",
        schema=Critique,
        system=REFLECTION_SYSTEM,
    )

## Step 3 (cont.) — Rewrite and loop

`rewrite` takes the document plus the structured `Critique` and produces an improved draft. `reflection_loop` alternates critique → rewrite until the score clears `quality_threshold` or we hit `max_iterations`. Because `critique` now returns a typed object, we read `feedback.score`, `feedback.weaknesses`, and `feedback.improvements` directly.

In [170]:
# rewrite applies the critique's fixes and returns an improved draft.
async def rewrite(document: str, feedback: Critique) -> str:
    """Rewrite the document, applying the critique's weaknesses and improvements."""
    weaknesses_text = ""
    for weakness in feedback.weaknesses:
        weaknesses_text += f"- {weakness}\n"
    improvements_text = ""
    for improvement in feedback.improvements:
        improvements_text += f"- {improvement}\n"
    return await acall(
        f"""# Context
You are improving a draft using a reviewer's critique. The original draft, the weaknesses to fix, and the improvements to apply are in the Input section.

# Task
Produce an improved version of the document.

# Constraints
- Apply every listed improvement and fix every listed weakness.
- Keep what already works; do not pad or add filler.
- Make it tighter, more specific, and better supported.

# Format
Return only the improved document text. No preamble, no commentary.

# Input
## Original document
{document}

## Weaknesses to fix
{weaknesses_text}

## Improvements to apply
{improvements_text}""",
        system=REWRITE_SYSTEM,
    )


# reflection_loop alternates critique -> rewrite until the score clears the
# threshold or we run out of iterations. It returns the final text plus a
# history list (one entry per round) so you can inspect what happened.
async def reflection_loop(
    document: str,
    quality_threshold: int = 8,
    max_iterations: int = 5,
) -> tuple[str, list[dict]]:
    """Critique -> rewrite until score >= threshold or max_iterations; return (final_text, history)."""
    history = []
    current = document
    prev_score=0

    for iteration in range(1, max_iterations + 1):
        print(f"  [Reflection {iteration}] Critiquing...")
        feedback = await critique(current)
        score = feedback.score
        print(f"  [Reflection {iteration}] Score: {score}/10")
        history.append({"iteration": iteration, "score": score, "feedback": feedback})

        if prev_score==score:
          break
        else:
          prev_score=score

        if score >= quality_threshold:
            print(f"  Quality threshold ({quality_threshold}) reached. Stopping.")
            break

        if iteration < max_iterations:
            print(f"  [Reflection {iteration}] Rewriting...")
            current = await rewrite(current, feedback)

    return current, history

## The full pipeline

`analyze_with_reflection` ties the three stages together: fan-out → fan-in → reflection. Each `print` is a checkpoint so you can watch the document evolve through critique iterations.

In [171]:
# analyze_with_reflection ties the whole pipeline together: fan-out -> fan-in -> reflection.
async def analyze_with_reflection(topic: str, quality_threshold: int = 8, strategy: str = "plain") -> dict:
    """Run the full fan-out -> fan-in -> reflection pipeline and return all artifacts."""
    print(f"\nTopic: {topic}")
    print("=" * 60)

    # Step 1: Fan-out -- run the three experts concurrently
    print("\n[1] Fan-out: parallel expert analysis")
    technical, business, ethics, ux = await fan_out(topic)
    print(f"    Technical Expert: {technical[:80]}...")
    print(f"    Business Strategist: {business[:80]}...")
    print(f"    Ethics Researcher: {ethics[:80]}...")
    print(f"    UX Researcher: {ux[:80]}...")

    # Step 2: Fan-in -- merge the three analyses into one draft
    #print("\n[2] Fan-in: synthesizing perspectives")
    #synthesis = await fan_in(topic, technical, business, ethics, ux)

    # Step 2: Fan-in Selection
    print(f"\n[2] Fan-in: synthesizing perspectives using strategy: '{strategy}'")
    if strategy == "structured":
        synthesis = await structured_synthesis(topic, technical, business, ethics, ux)
    elif strategy == "consensus":
        synthesis = await consensus_divergence_synthesis(topic, technical, business, ethics, ux)
    else:
        synthesis = await plain_fan_in(topic, technical, business, ethics, ux)

    # Step 3: Reflection -- critique and rewrite until it is good enough
    print("\n[3] Reflection loop")
    final_doc, history = await reflection_loop(synthesis, quality_threshold)

    return {
        "topic": topic,
        "technical": technical,
        "business": business,
        "ethics": ethics,
        "ux": ux,
        "initial_synthesis": synthesis,
        "final_document": final_doc,
        "reflection_history": history,
    }

## Run it

This is the cell that uses **top-level `await`** — the whole reason we avoided `asyncio.run`. In Colab the event loop is already running, so we simply `await` the pipeline coroutine. (Outside Colab, in a plain `.py` script, you would instead wrap this in `asyncio.run(...)`.)

In [172]:
TOPIC = (
    "The impact of agentic AI systems on software development workflows "
    "over the next five years"
)

result = await analyze_with_reflection(TOPIC, quality_threshold=9,strategy="consensus")

print("\n" + "=" * 60)
print("FINAL DOCUMENT")
print("=" * 60)
print(result["final_document"])

print("\n" + "=" * 60)
print("REFLECTION HISTORY")
print("=" * 60)
for entry in result["reflection_history"]:
    print(f"  Iteration {entry['iteration']}: score={entry['score']}/10")
    for w in entry["feedback"].weaknesses:
        print(f"    - Weakness: {w}")



Topic: The impact of agentic AI systems on software development workflows over the next five years

[1] Fan-out: parallel expert analysis
Fanning out to 4 experts in parallel...
  All 4 expert responses received.
    Technical Expert: Over the next five years, the shift from code-generation assistants to autonomou...
    Business Strategist: Agentic AI shifts software development from a labor-intensive "craft" model to a...
    Ethics Researcher: The integration of agentic AI into software development introduces a severe risk...
    UX Researcher: From a UX research perspective, the shift toward agentic AI necessitates a funda...

[2] Fan-in: synthesizing perspectives using strategy: 'consensus'
  Consensus/Divergence Fan-in complete. Synthesis: 387 words.

[3] Reflection loop
  [Reflection 1] Critiquing...
  [Reflection 1] Score: 7/10
  [Reflection 1] Rewriting...
  [Reflection 2] Critiquing...
  [Reflection 2] Score: 6/10
  [Reflection 2] Rewriting...
  [Reflection 3] Critiquing...


## Your turn (exercises)

1. **Add a fourth expert.** Write a new `ux_researcher(topic)` function (copy one of the experts and change the `# Role` persona), then plug it in three explicit places: the `asyncio.gather(...)` call in `fan_out`, `fan_out`'s return, and `fan_in`'s parameters and synthesis string. With separate functions you can see exactly where a new expert connects.
2. **Tune the quality threshold.** Run `analyze_with_reflection(TOPIC, quality_threshold=9)` and compare the final document and the number of reflection iterations against the `quality_threshold=8` run. Then try `quality_threshold=10` — does the loop ever satisfy it before `max_iterations`?

    ==> When quality_threshold=8, loop runs twice(2).
    When quality_threshold=9, loop runs thrice(3).
    When quality_threshold=10, loop fails while trying to have highest accuracy.

3. **Iteration budget.** Call `reflection_loop(synthesis, max_iterations=1)` vs `max_iterations=5` on the same synthesis. How much does the extra budget actually improve the score?

    ==> Increasing the max_iterations to 5 increases the chances for fixing more failures. But running repeatedly the failure fixtures rate decreases.
4. **Early stopping.** Add logic to `reflection_loop` that stops early if two consecutive critique scores don't improve (the rewrite isn't helping). Use the typed `Critique` objects already stored in `history`.
5. **Synthesis strategy.** Experiment with different `fan_in` strategies: plain concatenation, a structured merge with one section per perspective, or a consensus-vs-divergence layout. Which produces the best starting draft for reflection?
    
    ==> The consensus-vs-divergence layout produces the best starting draft. It gives clarity of which are strong points and which disagreeable points one has to work arround.

When you're done, save a copy (**File → Save a copy in Drive**) and submit your notebook via GitHub Classroom.